In [13]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import urllib.request
import re
from collections import Counter
import tqdm
import os

In [14]:
FILE_NAME = "text.txt"   # your own file in same folder

with open(FILE_NAME, 'r', encoding='utf-8') as f:
    text = f.read()

text = text.lower()
tokens = re.findall(r'\b\w+\b|[.,!?;]', text)

In [15]:
VOCAB_SIZE = 10000
MIN_FREQ = 5

# Count word frequencies
word_counts = Counter(tokens)
# Keep top words, reserve 0 for <PAD> and 1 for <UNK>
vocab = ['<PAD>', '<UNK>'] + [word for word, count in word_counts.most_common(VOCAB_SIZE - 2) if count >= MIN_FREQ]

word_to_idx = {word: idx for idx, word in enumerate(vocab)}
idx_to_word = {idx: word for word, idx in word_to_idx.items()}

print(f"Vocabulary size: {len(vocab)}")

# Convert all tokens to indices
encoded_text = [word_to_idx.get(token, word_to_idx['<UNK>']) for token in tokens]

Vocabulary size: 3233


In [16]:
SEQ_LEN = 30  # Look back 30 words to predict the 31st (much better than padding to 745)

class NextWordDataset(Dataset):
    def __init__(self, encoded_text, seq_len):
        self.data = encoded_text
        self.seq_len = seq_len

    def __len__(self):
        return len(self.data) - self.seq_len

    def __getitem__(self, idx):
        x = torch.tensor(self.data[idx : idx + self.seq_len], dtype=torch.long)
        y = torch.tensor(self.data[idx + self.seq_len], dtype=torch.long)
        return x, y

dataset = NextWordDataset(encoded_text, SEQ_LEN)
BATCH_SIZE = 128
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)

In [17]:
class NextWordLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers, dropout=0.3):
        super(NextWordLSTM, self).__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers

        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers,
                            batch_first=True, dropout=dropout if num_layers > 1 else 0)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, hidden=None):
        embed = self.embedding(x)
        out, hidden = self.lstm(embed, hidden)
        out = self.dropout(out)
        # We only need the output of the last time step for next-word prediction
        logits = self.fc(out[:, -1, :])
        return logits, hidden

# Initialize model
EMBED_DIM = 256
HIDDEN_DIM = 512
NUM_LAYERS = 2

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = NextWordLSTM(len(vocab), EMBED_DIM, HIDDEN_DIM, NUM_LAYERS, dropout=0.3).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=0.002)

In [ ]:
EPOCHS = 10
CLIP_MAX_NORM = 1.0  # Prevents exploding gradients in RNNs

print(f"Starting training on {device}...")
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    progress_bar = tqdm.tqdm(dataloader, desc=f"Epoch {epoch+1}/{EPOCHS}")

    for batch_x, batch_y in progress_bar:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)

        optimizer.zero_grad()
        logits, _ = model(batch_x)

        loss = criterion(logits, batch_y)
        loss.backward()

        # Gradient clipping is crucial for stable RNN/LSTM training
        torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP_MAX_NORM)

        optimizer.step()
        total_loss += loss.item()
        progress_bar.set_postfix(loss=loss.item())

    print(f"Epoch {epoch+1} Average Loss: {total_loss/len(dataloader):.4f}")

# Save the model
torch.save(model.state_dict(), "next_word_lstm.pth")
print("Model saved to next_word_lstm.pth")

Starting training on cuda...


Epoch 1/10: 100%|██████████| 1911/1911 [01:01<00:00, 31.31it/s, loss=4.35]


Epoch 1 Average Loss: 5.3342


Epoch 2/10: 100%|██████████| 1911/1911 [01:03<00:00, 30.11it/s, loss=4.94]


Epoch 2 Average Loss: 4.9261


Epoch 3/10: 100%|██████████| 1911/1911 [01:03<00:00, 30.10it/s, loss=5.07]


Epoch 3 Average Loss: 4.7103


Epoch 4/10: 100%|██████████| 1911/1911 [01:03<00:00, 30.09it/s, loss=6.24]


Epoch 4 Average Loss: 4.5692


Epoch 5/10:   5%|▍         | 90/1911 [00:02<01:00, 29.96it/s, loss=4.39]

In [ ]:
def generate_text(model, seed_text, num_words, seq_len, temperature=0.8):
    """
    Generates the next `num_words` given a `seed_text`.
    Temperature > 1.0 makes output more random, < 1.0 makes it more deterministic.
    """
    model.eval()
    words = re.findall(r'\b\w+\b|[.,!?;]', seed_text.lower())

    for _ in range(num_words):
        # Take the last `seq_len` words as context
        context = words[-seq_len:] if len(words) >= seq_len else words
        indices = [word_to_idx.get(w, word_to_idx['<UNK>']) for w in context]
        x = torch.tensor([indices], dtype=torch.long).to(device)

        with torch.no_grad():
            logits, _ = model(x)
            # Apply temperature scaling
            scaled_logits = logits / temperature
            probs = torch.softmax(scaled_logits, dim=-1)
            # Sample from the distribution (more natural than strict argmax)
            pred_idx = torch.multinomial(probs, 1).item()

        next_word = idx_to_word.get(pred_idx, '<UNK>')
        words.append(next_word)

    # Clean up spacing around punctuation
    result = ' '.join(words)
    result = re.sub(r'\s+([.,!?;])', r'\1', result)
    return result

In [ ]:
seed = "the king of"
print(f"\nSeed: '{seed}'")
print(f"Generated: {generate_text(model, seed, num_words=15, seq_len=SEQ_LEN, temperature=0.7)}")

seed = "to be or not to be"
print(f"\nSeed: '{seed}'")
print(f"Generated: {generate_text(model, seed, num_words=10, seq_len=SEQ_LEN, temperature=0.7)}")

In [ ]:
# Save everything together in one PyTorch checkpoint
checkpoint = {
    'model_state_dict': model.state_dict(),
    'word_to_idx': word_to_idx,
    'idx_to_word': idx_to_word,
    'seq_len': SEQ_LEN,
    'embed_dim': EMBED_DIM,
    'hidden_dim': HIDDEN_DIM,
    'num_layers': NUM_LAYERS
}

torch.save(checkpoint, 'next_word_lstm_full.pth')
print("Model and vocabulary bundled into next_word_lstm_full.pth")